In [3]:
from chemflow.dataset.molecule_data import (
    join_molecule_with_atoms,
    MoleculeData,
    PointCloud,
)

import torch
from torch_geometric.data import Batch


def test_join_molecule_with_atoms():
    print("=== Starting Test: join_molecule_with_atoms ===")

    # --- 1. Setup Data ---
    # Graph 0: 2 Nodes, 1 Edge (0-1)
    # Graph 1: 1 Node, 0 Edges
    mol_x = torch.tensor([[10.0], [10.0], [20.0]])  # Distinguishable features
    mol_edge_index = torch.tensor([[0, 1], [1, 0]], dtype=torch.long)
    mol_e = torch.tensor([1, 1], dtype=torch.long)  # Type 1 edges
    mol_batch = torch.tensor([0, 0, 1], dtype=torch.long)

    # Create Molecule Batch
    mol = MoleculeData(
        x=mol_x,
        edge_index=mol_edge_index,
        e=mol_e,
        batch=mol_batch,
        a=torch.zeros(3, 1),  # Dummy
        c=torch.zeros(3, 1),  # Dummy
    )

    # Atoms to add:
    # Batch 0: Adds 1 Atom
    # Batch 1: Adds 2 Atoms
    atom_x = torch.tensor([[100.0], [200.0], [200.0]])
    atom_batch = torch.tensor([0, 1, 1], dtype=torch.long)

    atoms = PointCloud(
        x=atom_x,
        batch=atom_batch,
        a=torch.zeros(3, 1),  # Dummy
        c=torch.zeros(3, 1),  # Dummy
    )

    # Distribution: [P(No_Edge), P(Type_1)]
    # We set P(Type_1) = 1.0 to force ALL possible new connections to form.
    # This makes verifying the topology deterministic.
    edge_distribution = torch.tensor([0.0, 1.0])

    # --- 2. Run Function ---
    new_mol = join_molecule_with_atoms(mol, atoms, edge_distribution)
    print(new_mol)

    # --- 3. Assertions ---

    # A. Check Node Counts
    # Expected: Graph 0 (2+1=3 nodes), Graph 1 (1+2=3 nodes) -> Total 6
    assert new_mol.num_nodes == 6, f"Expected 6 nodes, got {new_mol.num_nodes}"
    print("✅ Total node count correct.")

    # B. Check Features (Concatenation Order)
    # Dense batch usually pads, then un-pads. The order within a graph is preserved:
    # Graph 0: [Mol_Node_0, Mol_Node_1, Atom_Node_0] -> Features [10, 10, 100]
    # Graph 1: [Mol_Node_0, Atom_Node_0, Atom_Node_1] -> Features [20, 200, 200]
    # Note: The exact global order depends on how dense_to_sparse flattens,
    # but generally it is batch-major.

    # Let's check simply that we have 3 nodes with val 10, 1 with 20, 1 with 100, 2 with 200.
    vals = new_mol.x.squeeze()
    assert (vals == 10).sum() == 2
    assert (vals == 20).sum() == 1
    assert (vals == 100).sum() == 1
    assert (vals == 200).sum() == 2
    print("✅ Feature values preserved.")

    # C. Check Edges
    # Since we forced probability 1.0, we expect a quasi-clique behavior
    # (except strictly internal Mol-Mol edges that didn't exist before).

    # Graph 0 (Nodes A, B, C_new):
    # Existing: A-B
    # New: A-C_new, B-C_new (Mol-Atom)
    # Total edges = 1 (old) + 2 (new) = 3 undirected edges (6 directed indices)

    # Graph 1 (Node D, E_new, F_new):
    # Existing: None
    # New: D-E, D-F (Mol-Atom) AND E-F (Atom-Atom)
    # Total edges = 0 (old) + 3 (new) = 3 undirected edges (6 directed indices)

    total_edges = new_mol.edge_index.size(1)
    assert total_edges == 12, (
        f"Expected 12 directed edges (6 undirected), got {total_edges}"
    )
    print("✅ Total edge count correct (Old edges preserved + New edges created).")

    # D. Verify Edge Attributes
    # All edges should be Type 1
    assert (new_mol.e == 1).all(), "All edges should be type 1"
    print("✅ Edge attributes correct.")

    # E. Verify Batch Vector
    expected_batch = torch.tensor([0, 0, 0, 1, 1, 1], device=new_mol.x.device)
    # Note: We sort to ensure comparison works regardless of node permutation output
    # though dense_to_sparse usually preserves graph grouping.
    assert torch.equal(new_mol.batch.sort()[0], expected_batch), (
        "Batch vector is incorrect"
    )
    print("✅ Batch vector correct.")

    print("\n🎉 Test Passed!")


test_join_molecule_with_atoms()


=== Starting Test: join_molecule_with_atoms ===
MoleculeData(x=[6, 1], edge_index=[2, 17], a=[6, 1], c=[6, 1], e=[17], batch=[6])
✅ Total node count correct.
✅ Feature values preserved.


AssertionError: Expected 12 directed edges (6 undirected), got 17